# Tutorial 7: Measuring and Exporting

Detection tells you *where* colonies are. Measurement tells you *what they
are* — how big, how round, how bright. In this tutorial you will add
measurements to a pipeline, extract a DataFrame of colony features, and
export the results for downstream analysis.

**What you will learn:**

1. Add measurement operations to a pipeline
2. Use `pipeline.apply_and_measure()` to get a DataFrame
3. Understand the output columns
4. Export to CSV and Parquet

## Imports

In [ ]:
import phenotypic as pht
from phenotypic.data import load_yeast_plate
from phenotypic.enhance import GaussianBlur, CLAHE
from phenotypic.detect import OtsuDetector
from phenotypic.measure import MeasureSize, MeasureShape, MeasureIntensity

## Build a Pipeline with Measurements

The `meas` parameter accepts a list of measurement operations. Each one
extracts a different set of features from the detected colonies.

In [ ]:
plate = load_yeast_plate()

pipeline = pht.ImagePipeline(
    ops=[GaussianBlur(sigma=2.0), CLAHE(clip_limit=0.01), OtsuDetector()],
    meas=[MeasureSize(), MeasureShape(), MeasureIntensity()],
)

## Apply and Measure

`.apply_and_measure()` runs the full pipeline (enhance → detect → measure)
and returns a [pandas DataFrame](https://pandas.pydata.org/docs/user_guide/dsintro.html)
with one row per detected colony.

In [ ]:
df = pipeline.apply_and_measure(plate)
print(f"Measured {len(df)} colonies across {df.shape[1]} features")
df.head()

## Explore the Columns

Each measurement operation contributes its own set of columns. Let's see
what we got.

In [ ]:
print("All columns:")
for col in df.columns:
    print(f"  {col}")

Here are the highlights from each measurement:

**MeasureSize:**
- `Area` — colony size in pixels
- `IntegratedIntensity` — sum of grayscale pixel values

**MeasureShape:**
- `Circularity` — how round the colony is (1.0 = perfect circle)
- `Solidity` — ratio of colony area to convex hull area
- `Eccentricity` — elongation (0 = circular, approaching 1 = elongated)
- `MajorAxisLength` / `MinorAxisLength` — fitted ellipse axes

**MeasureIntensity:**
- `MeanIntensity` / `MedianIntensity` — average colony brightness
- `StandardDeviationIntensity` — variation within the colony
- `MinimumIntensity` / `MaximumIntensity` — intensity extremes

## Quick Statistics

Since the result is a standard pandas DataFrame, you can use all the usual
pandas methods to explore it.

In [ ]:
df[["Area", "Circularity", "MeanIntensity"]].describe()

## Export to CSV

For sharing with collaborators or importing into spreadsheet software,
export to CSV.

In [ ]:
df.to_csv("colony_measurements.csv")
print("Saved to colony_measurements.csv")

## Export to Parquet

For large datasets, [Parquet](https://parquet.apache.org/) is more
efficient — it is compressed, preserves column types, and loads much
faster than CSV.

In [ ]:
df.to_parquet("colony_measurements.parquet")
print("Saved to colony_measurements.parquet")

## Clean Up

In [ ]:
import os
os.remove("colony_measurements.csv")
os.remove("colony_measurements.parquet")

## Summary

You have extracted colony features and exported them for analysis:

- **`meas=[MeasureSize(), MeasureShape(), MeasureIntensity()]`** — add measurements to a pipeline
- **`pipeline.apply_and_measure(plate)`** — run the full pipeline and get a DataFrame
- **`.to_csv()`** / **`.to_parquet()`** — export for downstream tools

The result is a standard pandas DataFrame, so you can filter, group, plot,
and analyze it with any tool in the Python ecosystem.

**Next up:** [Tutorial 8: Using Prefab Pipelines](08_using_prefab_pipelines.ipynb) —
discover PhenoTypic's pre-built pipelines for common organisms and plate types.